## Exercises on Policy Gradient Methods

These paper-and-pencil exercises reinforce Chapter 08: the softmax policy parameterisation, the score function $\nabla_\theta\log\pi_\theta(a\mid s)$, the likelihood-ratio (log-derivative) trick, the policy-gradient theorem and the REINFORCE estimator, the role of causality and of a baseline in reducing variance, entropy regularisation, and the translation of the estimator into a loss function. (The actor–critic material — bootstrapped critics, $n$-step advantages, GAE — is treated in the Chapter 09 exercises.) Notation: $\pi_\theta(a\mid s)$ is the parameterised policy, $J(\theta)$ the objective, $G_t$ the return from time $t$, $V(s;\phi)$ the learned baseline, $H$ the entropy and $\beta$ its coefficient.

### Exercise 8.1 — The softmax policy and its score function

A two-action softmax policy uses linear preferences $h(s,a)=\theta^\top x(s,a)$ with one-hot features $x(s,a_1)=(1,0)$, $x(s,a_2)=(0,1)$ and parameters $\theta=(1,0)$.

1. Compute the action probabilities $\pi_\theta(a\mid s)$.
2. Derive the score function $\nabla_\theta\log\pi_\theta(a\mid s)$ for the softmax and evaluate it for $a=a_1$.

**Step 1 — Action probabilities.** $\ \pi_\theta(a\mid s)=\dfrac{e^{h(s,a)}}{\sum_b e^{h(s,b)}}$, with $h(s,a_1)=\theta_1=1,\ h(s,a_2)=\theta_2=0$:

$\displaystyle \pi(a_1\mid s)=\frac{e^{1}}{e^{1}+e^{0}} = 0.7311, \qquad \pi(a_2\mid s)=0.2689.$

**Step 2 — Score function.** Taking $\log$ and differentiating the softmax gives the classic "feature minus expected feature":

$\displaystyle \nabla_\theta \log\pi_\theta(a\mid s) = x(s,a) - \sum_{b}\pi_\theta(b\mid s)\,x(s,b) = x(s,a) - \mathbb{E}_{b\sim\pi}[x(s,b)].$

Here $\mathbb{E}_\pi[x] = 0.7311(1,0)+0.2689(0,1) = (0.7311, 0.2689)$, so

$\displaystyle \nabla_\theta \log\pi_\theta(a_1\mid s) = (1,0) - (0.7311, 0.2689) = (0.2689, -0.2689).$

**Step 3 — Reading it.** The positive first component means increasing $\theta_1$ raises $\pi(a_1)$; the negative second component means it lowers $\pi(a_2)$. The score points in the direction of parameter space that makes the *taken* action more likely.

**Key concept**

For a softmax policy the score is the **feature of the chosen action minus the policy-averaged feature**. Policy-gradient methods move $\theta$ along this direction, weighted by how good the outcome was.

### Exercise 8.2 — The expected score is zero (why baselines are free)

Prove that the expected score under the policy vanishes, $\ \mathbb{E}_{a\sim\pi_\theta}[\nabla_\theta\log\pi_\theta(a\mid s)] = 0$, and verify it numerically for the policy of Exercise 8.1.

**Step 1 — General proof.** Using $\pi\,\nabla\log\pi = \nabla\pi$ (the log-derivative identity) and $\sum_a\pi_\theta(a\mid s)=1$:

$\displaystyle \mathbb{E}_{a\sim\pi}\big[\nabla_\theta\log\pi_\theta(a\mid s)\big] = \sum_a \pi_\theta(a\mid s)\,\nabla_\theta\log\pi_\theta(a\mid s) = \sum_a \nabla_\theta \pi_\theta(a\mid s) = \nabla_\theta \sum_a \pi_\theta(a\mid s) = \nabla_\theta 1 = 0.$

**Step 2 — Numerical check** (Exercise 8.1 policy). The scores are $\nabla\log\pi(a_1)=(0.2689, -0.2689)$ and $\nabla\log\pi(a_2)=(0,1)-(0.7311, 0.2689)=(-0.7311, 0.7311)$. Then

$\displaystyle \pi(a_1)\,(0.2689, -0.2689) + \pi(a_2)\,(-0.7311, 0.7311) = 0.7311(0.2689, -0.2689) + 0.2689(-0.7311, 0.7311) = (0, 0). \checkmark$

**Step 3 — Consequence.** Because the expected score is zero, subtracting **any** quantity $b(s)$ that does not depend on the action leaves the policy gradient unchanged in expectation:

$\displaystyle \mathbb{E}_a\big[\nabla_\theta\log\pi_\theta(a\mid s)\,b(s)\big] = b(s)\,\mathbb{E}_a\big[\nabla_\theta\log\pi_\theta(a\mid s)\big] = 0.$

**Key concept**

The zero-expected-score identity is the theoretical licence for **baselines**: we may subtract a state-dependent baseline $b(s)$ from the return to reduce variance *without introducing bias*. This is exploited in the next exercise and underpins VPG and the actor–critic methods of Chapter 09.

### Exercise 8.3 — A single REINFORCE update (bandit case)

A one-state (bandit) softmax policy over two actions starts from $\theta=(0,0)$ (so $\pi$ is uniform), with one-hot features as in Exercise 8.1. The agent samples action $a_1$ and receives reward $R=1$. Using the REINFORCE gradient estimate $\hat g = R\,\nabla_\theta\log\pi_\theta(a_1\mid s)$ and learning rate $\alpha=0.2$, perform one update and report the new probability of $a_1$.

**Step 1 — Initial policy.** $\theta=(0,0)\Rightarrow \pi(a_1)=\pi(a_2)=0.5$, so $\mathbb{E}_\pi[x]=(0.5,0.5)$.

**Step 2 — Gradient estimate.** Score of the taken action $a_1$: $\ \nabla_\theta\log\pi(a_1)=x(a_1)-\mathbb{E}_\pi[x]=(1,0)-(0.5,0.5)=(0.5,-0.5).$ With $R=1$:

$\displaystyle \hat g = R\,\nabla_\theta\log\pi(a_1) = (0.5,-0.5).$

**Step 3 — Update.** $\ \theta \leftarrow \theta + \alpha\,\hat g = (0,0) + 0.2(0.5,-0.5) = (0.1, -0.1).$

**Step 4 — New probability.** $\ \pi(a_1) = \dfrac{e^{0.1}}{e^{0.1}+e^{-0.1}} = 0.5498$, up from $0.5$.

Because the reward was positive, the update **increased** the probability of the sampled action.

**Key concept**

REINFORCE performs *likelihood-ratio* gradient ascent: it pushes up the log-probability of actions that led to high return and pushes down those that led to low return. With a positive reward and a uniform start, $\pi(a_1)$ rises.

### Exercise 8.4 — REINFORCE on a two-step episode

An agent uses a **tabular softmax** policy: for each state, two actions with preference parameters $h(s,a)=\theta(s,a)$, initialised to zero so that every action has probability $0.5$. It runs one episode, $\gamma=1$:

$\displaystyle s_0 \xrightarrow{a_0,\ R_1=0} s_1 \xrightarrow{a_1,\ R_2=1} \text{terminal}.$

Using the **causality** form of the policy gradient (each action weighted by the return *from that step onward*) and $\alpha=0.1$, compute the updates to the preferences of the taken actions and the resulting $\pi(a_0\mid s_0)$.

**Step 1 — Causal returns.** $\ G_0 = R_1+R_2 = 0+1 = 1$ (weights $a_0$), $\quad G_1 = R_2 = 1$ (weights $a_1$).

**Step 2 — Scores for a tabular 2-action softmax.** For the taken action, $\nabla_{\theta(s,a)}\log\pi(a\mid s)=1-\pi(a\mid s)$; for the other action it is $-\pi(\cdot\mid s)$. With every $\pi=0.5$:

$\displaystyle \nabla\log\pi(a_0\mid s_0) = 1-0.5 = 0.5, \qquad \nabla\log\pi(a_1\mid s_1) = 1-0.5 = 0.5.$

**Step 3 — Updates** $\theta(s,a)\leftarrow\theta(s,a)+\alpha\,G_t\,\nabla\log\pi$:

$\displaystyle \theta(s_0,a_0) \leftarrow 0 + 0.1\,(1)(0.5) = 0.05, \qquad \theta(s_0,\text{other}) \leftarrow 0 + 0.1\,(1)(-0.5) = -0.05,$

and identically for $s_1$.

**Step 4 — New probability.** $\ \pi(a_0\mid s_0) = \dfrac{e^{0.05}}{e^{0.05}+e^{-0.05}} = 0.525$, up from $0.5$; the taken action in each visited state becomes more likely.

**Key concept**

REINFORCE credits each action with the return that *follows* it (causality: past rewards cannot depend on the current action). Because the whole episode's return here is positive, every action taken is reinforced — which, without a baseline, is exactly the high-variance behaviour the next exercise addresses.

### Exercise 8.5 — A baseline reduces variance (integrative)

A one-state policy has two actions with equal probability $\pi=(0.5,0.5)$; the (scalar) score of $a_1$ is $+0.5$ and of $a_2$ is $-0.5$ (as in Exercise 8.3). The returns are $q(a_1)=11$ and $q(a_2)=9$ (both positive). Consider the single-sample gradient estimator $\hat g = X(a)\,\nabla\log\pi(a)$, where the action is sampled from $\pi$.

1. With $X(a)=q(a)$ (no baseline), compute $\mathbb{E}[\hat g]$ and $\mathrm{Var}[\hat g]$.
2. With $X(a)=A(a)=q(a)-b$ and baseline $b=v=\mathbb{E}_\pi[q]$, recompute both.
3. Interpret.

**Step 1 — No baseline.** The estimator takes two values: pick $a_1$ ($\hat g=11(0.5)=5.5$) or $a_2$ ($\hat g=9(-0.5)=-4.5$), each with prob. $0.5$.

$\displaystyle \mathbb{E}[\hat g] = 0.5(5.5)+0.5(-4.5) = 0.5, \qquad \mathrm{Var}[\hat g] = 0.5(5.5^2+4.5^2) - 0.5^2 = 25.25-0.25 = 25.$

**Step 2 — With baseline** $b=v=0.5(11)+0.5(9)=10$, so advantages $A(a_1)=+1,\ A(a_2)=-1$. Now $\hat g$ takes: $a_1\!:1(0.5)=0.5$, $a_2\!: (-1)(-0.5)=0.5$ — the **same** value in both cases.

$\displaystyle \mathbb{E}[\hat g] = 0.5 \ (\text{unchanged}), \qquad \mathrm{Var}[\hat g] = 0.5(0.5^2+0.5^2)-0.5^2 = 0.$

**Step 3 — Interpretation.** The baseline left the *expected* gradient unchanged ($0.5$, confirming unbiasedness from Exercise 8.2) but collapsed the variance from $25$ to $0$. Intuitively, when all raw returns are positive the estimator keeps pushing every action up and only the *differences* carry signal; subtracting the mean ($v$) centres the returns so better-than-average actions get positive weight and worse-than-average get negative weight.

**Key concept**

Subtracting a baseline (ideally the state value $v_\pi(s)$, giving the **advantage** $a_\pi(s,a)=q_\pi(s,a)-v_\pi(s)$) is the single most important variance-reduction technique in policy-gradient methods. It is *free of bias* precisely because the expected score is zero — the bridge from REINFORCE to actor–critic.

### Exercise 8.6 — Entropy regularisation

For a two-action policy with $\pi=(p,\,1-p)$, the entropy is $H(\pi)=-\big[p\ln p + (1-p)\ln(1-p)\big]$ (in nats).

1. Compute $H$ for the uniform policy $p=0.5$ and for the near-deterministic policy $p=0.9$.
2. Show that entropy is maximised at $p=0.5$, and explain how adding $\beta H(\pi)$ to the objective of VPG affects learning.

**Step 1 — Evaluate.**

$\displaystyle H(0.5,0.5) = -\big[0.5\ln 0.5 + 0.5\ln 0.5\big] = \ln 2 = 0.6931\ \text{nats},$
$\displaystyle H(0.9,0.1) = -\big[0.9\ln 0.9 + 0.1\ln 0.1\big] = 0.3251\ \text{nats}.$

The near-deterministic policy has **lower** entropy.

**Step 2 — Maximum at uniform.** Setting $\tfrac{dH}{dp} = -\ln p + \ln(1-p) = \ln\tfrac{1-p}{p} = 0$ gives $p=0.5$; the second derivative $-\tfrac1p-\tfrac1{1-p}<0$ confirms a maximum. So entropy is largest for the uniform policy and $0$ for a deterministic one.

**Step 3 — Effect of the bonus.** Maximising $J(\theta)+\beta H(\pi_\theta)$ rewards *keeping the policy spread out*. Early in training this **discourages premature collapse** onto one action (which would set $H\to0$: the other actions would never be sampled again, their scores never updated, and the policy could not recover); as learning proceeds, the return term dominates and the policy is allowed to sharpen. $\beta$ controls the strength of this pressure: too large and the policy stays near uniform and never exploits, $\beta=0$ recovers the unregularised algorithm.

**Key concept**

An entropy bonus is a built-in exploration incentive for policy-gradient methods: it penalises over-confident policies, keeping action probabilities diverse long enough to discover good actions — the on-policy counterpart of $\varepsilon$-greedy and optimistic initialisation (Chapter 04).

### Exercise 8.7 — Causality: past rewards are pure noise

An episode has two steps and $\gamma=1$. In $s_0$ the agent takes $a_0$ and receives $R_1=1$ (deterministic). In $s_1$ a tabular softmax policy with preferences $h(s_1,\cdot)=\theta(s_1,\cdot)=(0,0)$ chooses $L$ or $R$ with probability $0.5$ each; the reward is $R_2=+1$ after $L$ and $R_2=-1$ after $R$, then the episode ends. Consider the single-sample gradient estimate for the preference $\theta(s_1,L)$, using as weight either the **full return** $G(\tau)=R_1+R_2$ or the **reward-to-go** $G_1=R_2$.

1. Compute the score $\partial\log\pi(A_1\mid s_1)/\partial\theta(s_1,L)$ for $A_1=L$ and $A_1=R$.
2. Compute mean and variance of the estimate with the full return.
3. Compute mean and variance of the estimate with the reward-to-go, and interpret.

**Step 1 — Scores.** For a tabular softmax, $\partial\log\pi(a\mid s)/\partial\theta(s,L) = \mathbb{1}_{[a=L]}-\pi(L\mid s)$, so with $\pi(L\mid s_1)=0.5$:

$\displaystyle A_1=L:\ \ 1-0.5=+0.5, \qquad A_1=R:\ \ 0-0.5=-0.5.$

**Step 2 — Full return.** $G(\tau)=R_1+R_2$ equals $2$ after $L$ and $0$ after $R$. The estimate $\hat g = G(\tau)\cdot\text{score}$ takes the values $2(0.5)=1$ (prob. $0.5$) and $0\cdot(-0.5)=0$ (prob. $0.5$):

$\displaystyle \mathbb{E}[\hat g]=0.5(1)+0.5(0)=0.5, \qquad \mathrm{Var}[\hat g]=0.5(1^2+0^2)-0.5^2=0.25.$

**Step 3 — Reward-to-go.** $G_1=R_2$ equals $+1$ after $L$ and $-1$ after $R$. The estimate takes the values $1(0.5)=0.5$ and $(-1)(-0.5)=0.5$ — the same value in both cases:

$\displaystyle \mathbb{E}[\hat g]=0.5 \ (\text{unchanged}), \qquad \mathrm{Var}[\hat g]=0.$

The difference between the two estimators is the term $R_1\cdot\text{score}$, whose expectation is $R_1\,\mathbb{E}_{A_1\sim\pi}[\text{score}] = R_1\cdot 0 = 0$: the past reward multiplies a zero-mean quantity and contributes only variance. Note that $R_1$ acts exactly like a constant baseline with the opposite sign.

**Key concept**

An action cannot influence the rewards that preceded it, and the policy-gradient estimator knows it *in expectation* only: including past rewards leaves the mean unchanged but adds variance. Weighting each action by its **reward-to-go** $G_t$ (causality) is therefore a free variance reduction, by the same zero-expected-score identity that justifies baselines.

### Exercise 8.8 — From the estimator to the loss function

The REINFORCE code minimises, for one episode of length $T$, the loss $\ \mathcal{L}(\theta) = -\dfrac1T\sum_{t=0}^{T-1} G_t\,\log\pi_\theta(A_t\mid S_t)$.

1. Show that a gradient-descent step on $\mathcal{L}$ is a gradient-ascent step on $J$, and find the effective step size.
2. Explain why the returns $G_t$ (and, in VPG, the advantages) must be treated as constants when differentiating — what would go wrong in VPG without `detach()`?
3. An episode has returns $G=(3,2,1)$. Compute $\mathcal{L}$ when the taken actions had probability $0.5$ each, and when they had probability $0.9$ each. What does the comparison say about using $\mathcal{L}$ to monitor learning?

**Step 1 — Sign and step size.** With the $G_t$ held fixed,

$\displaystyle \nabla_\theta\mathcal{L} = -\frac1T\sum_{t=0}^{T-1} G_t\,\nabla_\theta\log\pi_\theta(A_t\mid S_t) = -\frac1T\,\hat g,$

where $\hat g=\sum_t G_t\nabla_\theta\log\pi_\theta(A_t\mid S_t)$ is the (single-trajectory) policy-gradient estimate. The descent step $\theta\leftarrow\theta-\alpha\nabla_\theta\mathcal{L} = \theta+\frac{\alpha}{T}\,\hat g$ is an **ascent** step on $J$ with effective step size $\alpha/T$: the division by $T$ keeps the size of the update comparable between short and long episodes.

**Step 2 — Constants.** The policy-gradient theorem gives $\nabla_\theta J = \mathbb{E}[\sum_t \nabla_\theta\log\pi_\theta\cdot G_t]$: the weights $G_t$ are *observed numbers*, not functions of $\theta$, so differentiating them would compute something else. In VPG the weight is $A_t=G_t-V(S_t;\phi)$, which depends on the critic parameters $\phi$. Without `detach()`, the policy loss $-\frac1T\sum_t A_t\log\pi_\theta$ would also produce a gradient with respect to $\phi$, equal to $+\frac1T\sum_t \log\pi_\theta(A_t\mid S_t)\,\nabla_\phi V(S_t;\phi)$: the value network would be pushed to *raise* $V$ wherever $\log\pi$ is very negative (improbable actions), an objective that has nothing to do with predicting returns, and that would corrupt the baseline. The baseline must be fitted only by its own regression loss.

**Step 3 — The loss is not a performance measure.** With $\pi=0.5$ for every taken action, $\log\pi=\ln 0.5=-0.6931$:

$\displaystyle \mathcal{L} = -\tfrac13\,(3+2+1)(-0.6931) = 1.386.$

With $\pi=0.9$, $\log\pi=\ln 0.9=-0.1054$:

$\displaystyle \mathcal{L} = -\tfrac13\,(6)(-0.1054) = 0.211.$

The returns — hence the performance — are identical, yet the loss dropped by a factor of about $6.5$. $\mathcal{L}$ depends on how *probable* the taken actions were, not on how *good* the episode was; it can also become negative when returns are negative. Unlike the mean squared TD error of Chapter 07, its value carries no information about progress: learning must be monitored through the evaluation score.

**Key concept**

The policy-gradient loss is only a *device to obtain the right gradient*: minus the return-weighted log-likelihood of the actions taken, with the weights detached. Its sign turns ascent on $J$ into descent, its $1/T$ rescales the step, and its numerical value says nothing about how well the agent is doing.